# Deteksi Inkonsistensi Sentimen dan Rating pada Review Aplikasi Mobile Indonesia

Notebook ini mencakup dua bagian utama:
1. **Data Collection** - Scraping review dari Google Play Store
2. **Preprocessing** - Pembersihan dan transformasi teks

## Bagian 1: Data Collection

### 1.1 Instalasi Library

In [22]:
import subprocess
import sys

packages = [
    'google-play-scraper',
    'pandas',
    'numpy',
    'langdetect',
    'tqdm'
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Semua library berhasil diinstall.')

Installing google-play-scraper...
Installing pandas...
Installing numpy...
Installing langdetect...
Installing tqdm...
Semua library berhasil diinstall.


### 1.2 Import Library

In [23]:
import pandas as pd
import numpy as np
import re
import time
import warnings
from datetime import datetime

from google_play_scraper import reviews, Sort
from langdetect import detect, LangDetectException
from tqdm import tqdm

warnings.filterwarnings('ignore')

print('Semua library berhasil diimport.')
print(f'Pandas version  : {pd.__version__}')
print(f'Numpy version   : {np.__version__}')

Semua library berhasil diimport.
Pandas version  : 2.3.2
Numpy version   : 2.3.3


### 1.3 Konfigurasi Aplikasi Target

In [24]:
APPS = [
    {'name': 'Gojek',      'app_id': 'com.gojek.app'},
    {'name': 'Tokopedia',  'app_id': 'com.tokopedia.tkpd'},
    {'name': 'Shopee',     'app_id': 'com.shopee.id'},
    {'name': 'DANA',       'app_id': 'id.dana'},
    {'name': 'BCA Mobile', 'app_id': 'com.bca'},
]

SCRAPE_COUNT   = 5000
MAX_RETRY      = 3
DELAY_SECONDS  = 2
MIN_TOTAL      = 10000
EXTRA_COUNT    = 500

print(f'Total aplikasi target : {len(APPS)}')
print(f'Review per aplikasi   : {SCRAPE_COUNT:,}')
print(f'Target total review   : {SCRAPE_COUNT * len(APPS):,} (sebelum filtering)')

Total aplikasi target : 5
Review per aplikasi   : 5,000
Target total review   : 25,000 (sebelum filtering)


### 1.4 Fungsi Scraping dengan Retry Logic

In [25]:
def scrape_app_reviews(app_name, app_id, count=2500, max_retry=3):
    """
    Scrape review dari satu aplikasi dengan retry logic.
    Mengembalikan list of dict atau list kosong jika gagal.
    """
    for attempt in range(1, max_retry + 1):
        try:
            print(f'  Scraping {app_name} ({app_id}) - percobaan {attempt}/{max_retry}...')
            result, _ = reviews(
                app_id,
                lang='id',
                country='id',
                sort=Sort.NEWEST,
                count=count
            )

            records = []
            for r in result:
                records.append({
                    'app':          app_name,
                    'app_id':       app_id,
                    'username':     r.get('userName', ''),
                    'rating':       r.get('score', np.nan),
                    'text':         r.get('content', ''),
                    'thumbs_up':    r.get('thumbsUpCount', 0),
                    'date':         r.get('at', None),
                    'reply':        r.get('replyContent', ''),
                })

            print(f'  Berhasil mengambil {len(records):,} review dari {app_name}.')
            return records

        except Exception as e:
            print(f'  Gagal (percobaan {attempt}): {e}')
            if attempt < max_retry:
                print(f'  Menunggu {DELAY_SECONDS} detik sebelum retry...')
                time.sleep(DELAY_SECONDS)

    print(f'  PERINGATAN: Gagal scraping {app_name} setelah {max_retry} percobaan.')
    return []


print('Fungsi scraping siap.')

Fungsi scraping siap.


### 1.5 Proses Scraping Utama

In [26]:
all_records = []
app_record_counts = {}

print('=' * 60)
print('MEMULAI PROSES SCRAPING')
print('=' * 60)

for app in tqdm(APPS, desc='Progress scraping'):
    print(f'\nMulai scraping: {app["name"]}')
    records = scrape_app_reviews(
        app_name=app['name'],
        app_id=app['app_id'],
        count=SCRAPE_COUNT,
        max_retry=MAX_RETRY
    )
    all_records.extend(records)
    app_record_counts[app['name']] = len(records)
    print(f'Total review terkumpul sejauh ini: {len(all_records):,}')
    time.sleep(DELAY_SECONDS)

print('\n' + '=' * 60)
print(f'SCRAPING SELESAI. Total raw review: {len(all_records):,}')
print('=' * 60)

for app_name, count in app_record_counts.items():
    print(f'  {app_name:<15}: {count:,} review')

MEMULAI PROSES SCRAPING


Progress scraping:   0%|          | 0/5 [00:00<?, ?it/s]


Mulai scraping: Gojek
  Scraping Gojek (com.gojek.app) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari Gojek.
Total review terkumpul sejauh ini: 5,000


Progress scraping:  20%|██        | 1/5 [00:04<00:18,  4.56s/it]


Mulai scraping: Tokopedia
  Scraping Tokopedia (com.tokopedia.tkpd) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari Tokopedia.
Total review terkumpul sejauh ini: 10,000


...


Mulai scraping: DANA
  Scraping DANA (id.dana) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari DANA.
Total review terkumpul sejauh ini: 20,000


Progress scraping:  80%|████████  | 4/5 [00:20<00:05,  5.15s/it]


Mulai scraping: BCA Mobile
  Scraping BCA Mobile (com.bca) - percobaan 1/3...
  Berhasil mengambil 5,000 review dari BCA Mobile.
Total review terkumpul sejauh ini: 25,000


Progress scraping: 100%|██████████| 5/5 [00:26<00:00,  5.37s/it]


SCRAPING SELESAI. Total raw review: 25,000
  Gojek          : 5,000 review
  Tokopedia      : 5,000 review
  Shopee         : 5,000 review
  DANA           : 5,000 review
  BCA Mobile     : 5,000 review


### 1.6 Simpan Raw Data

In [27]:
df_raw = pd.DataFrame(all_records)

print(f'Shape raw DataFrame  : {df_raw.shape}')
print(f'Kolom yang tersedia  : {list(df_raw.columns)}')
print('\nContoh data (5 baris pertama):')
display(df_raw.head())

df_raw.to_csv('raw_reviews.csv', index=False, encoding='utf-8-sig')
print('\nRaw data berhasil disimpan ke raw_reviews.csv')

Shape raw DataFrame  : (25000, 8)
Kolom yang tersedia  : ['app', 'app_id', 'username', 'rating', 'text', 'thumbs_up', 'date', 'reply']

Contoh data (5 baris pertama):


,app,app_id,username,rating,text,thumbs_up,date,reply
0,Gojek,com.gojek.app,Pengguna Google,4,bagus banget,0,2026-05-30 17:01:26,None
1,Gojek,com.gojek.app,Pengguna Google,5,saya suka gojek dan semua dr aplikasi ini semo...,0,2026-05-30 16:59:41,None
2,Gojek,com.gojek.app,Pengguna Google,1,"ga pemilik aplikasi nya ,ga pengiklan nya ,ga ...",1,2026-05-30 16:41:25,None
3,Gojek,com.gojek.app,Pengguna Google,5,"katanya potongan cuma 8%,ke driver. gw dapet o...",1,2026-05-30 16:39:05,None
4,Gojek,com.gojek.app,Pengguna Google,5,"oke, sangat membantu",0,2026-05-30 16:34:18,None



Raw data berhasil disimpan ke raw_reviews.csv


## Bagian 2: Preprocessing

### 2.1 Load Data dan Inisialisasi

In [28]:
df = df_raw.copy()

print(f'Data dimuat: {len(df):,} baris, {df.shape[1]} kolom')
print('\nInfo kolom:')
print(df.dtypes)
print('\nJumlah missing values per kolom:')
print(df.isnull().sum())

Data dimuat: 25,000 baris, 8 kolom

Info kolom:
app                  object
app_id               object
username             object
rating                int64
text                 object
thumbs_up             int64
date         datetime64[ns]
reply                object
dtype: object

Jumlah missing values per kolom:
app              0
app_id           0
username         0
rating           0
text             0
thumbs_up        0
date             0
reply        12610
dtype: int64


### 2.2 Filter Review Kosong dan Terlalu Pendek

In [29]:
before = len(df)

# Hapus review dengan teks NaN atau kosong
df = df[df['text'].notna()]
df = df[df['text'].str.strip() != '']

# Hapus review dengan kurang dari 5 kata
df['_word_count_temp'] = df['text'].apply(lambda x: len(str(x).split()))
df = df[df['_word_count_temp'] >= 5]
df = df.drop(columns=['_word_count_temp'])
df = df.reset_index(drop=True)

after = len(df)
print(f'Filter teks kosong/pendek:')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} review')

Filter teks kosong/pendek:
  Sebelum : 25,000
  Sesudah : 13,654
  Dihapus : 11,346 review


### 2.3 Filter Bahasa Indonesia

In [30]:
before = len(df)

def detect_language(text):
    try:
        lang = detect(str(text))
        return lang
    except LangDetectException:
        return 'unknown'
    except Exception:
        return 'unknown'

tqdm.pandas(desc='Deteksi bahasa')
df['lang'] = df['text'].progress_apply(detect_language)

lang_dist = df['lang'].value_counts()
print('\nDistribusi bahasa yang terdeteksi:')
print(lang_dist.head(10))

df = df[df['lang'] == 'id'].reset_index(drop=True)

after = len(df)
print(f'\nFilter bahasa Indonesia:')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} review (bukan bahasa Indonesia)')

Deteksi bahasa: 100%|██████████| 13654/13654 [01:21<00:00, 167.95it/s]


Distribusi bahasa yang terdeteksi:
lang
id    12793
tl      341
de      184
en      104
et       34
so       26
hr       25
no       16
fi       13
sl       12
Name: count, dtype: int64

Filter bahasa Indonesia:
  Sebelum : 13,654
  Sesudah : 12,793
  Dihapus : 861 review (bukan bahasa Indonesia)


### 2.4 Normalisasi Teks

In [31]:
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F926-\U0001F937"
    "\U00010000-\U0010FFFF"
    "♀-♂"
    "☀-⭕"
    "‍⏏⏩⌚️〰"
    "]+",
    flags=re.UNICODE
)

def normalize_text(text):
    text = str(text)
    # Lowercase
    text = text.lower()
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)
    # Hapus emoji
    text = EMOJI_PATTERN.sub('', text)
    # Hapus karakter yang berulang lebih dari 2 kali berturut-turut
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Simpan teks asli sebelum normalisasi
df['text_original'] = df['text']

tqdm.pandas(desc='Normalisasi teks')
df['text'] = df['text'].progress_apply(normalize_text)

print('Normalisasi teks selesai.')
print('\nContoh sebelum dan sesudah normalisasi (5 sampel):')
sample_idx = df.sample(5, random_state=42).index
for i in sample_idx:
    print(f'  Asli  : {df.loc[i, "text_original"][:80]}')
    print(f'  Norma : {df.loc[i, "text"][:80]}')
    print()

Normalisasi teks: 100%|██████████| 12793/12793 [00:00<00:00, 35997.36it/s]

Normalisasi teks selesai.

Contoh sebelum dan sesudah normalisasi (5 sampel):
  Asli  : lain kali jadi cs yang bener lah....bukanya mempermudah malah bikin pening kepal
  Norma : lain kali jadi cs yang bener lah..bukanya mempermudah malah bikin pening kepala

  Asli  : semoga amanah Dan selalu memudahkan dalam transaksi
  Norma : semoga amanah dan selalu memudahkan dalam transaksi

  Asli  : ok banget. jam berapa saja selalu ada
  Norma : ok banget. jam berapa saja selalu ada

  Asli  : eh apk anj,gua kalo lagi buka web gausah elu dong yv muncuk iklannya anjj,jadi n
  Norma : eh apk anj,gua kalo lagi buka web gausah elu dong yv muncuk iklannya anjj,jadi n

  Asli  : ÀPLIKASI ini sangat membantu kita untuk belanja
  Norma : àplikasi ini sangat membantu kita untuk belanja



### 2.5 Hapus Duplikat

In [32]:
before = len(df)

df = df.drop_duplicates(subset=['text', 'app'], keep='first').reset_index(drop=True)

after = len(df)
print(f'Hapus duplikat (berdasarkan text + app):')
print(f'  Sebelum : {before:,}')
print(f'  Sesudah : {after:,}')
print(f'  Dihapus : {before - after:,} duplikat')

Hapus duplikat (berdasarkan text + app):
  Sebelum : 12,793
  Sesudah : 12,784
  Dihapus : 9 duplikat


### 2.6 Tambah Kolom word_count

In [33]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

print('Kolom word_count berhasil ditambahkan.')
print(f'\nStatistik word_count:')
print(df['word_count'].describe().round(2))

Kolom word_count berhasil ditambahkan.

Statistik word_count:
count    12784.00
mean        19.06
std         15.94
min          4.00
25%          8.00
50%         14.00
75%         24.00
max         98.00
Name: word_count, dtype: float64


### 2.7 Validasi Total Review dan Scraping Tambahan

In [34]:
iteration = 0

while len(df) < MIN_TOTAL:
    iteration += 1
    print('=' * 60)
    print(f'PERINGATAN: Total review setelah preprocessing = {len(df):,}')
    print(f'Kurang dari minimum {MIN_TOTAL:,}. Memulai iterasi tambahan ke-{iteration}...')
    print('=' * 60)

    # Identifikasi 2 aplikasi dengan review paling sedikit
    counts_per_app = df['app'].value_counts()
    apps_sorted    = counts_per_app.sort_values().head(2).index.tolist()

    print(f'\nAplikasi yang akan di-scrape tambahan: {apps_sorted}')

    new_records = []
    for app_name in apps_sorted:
        app_cfg = next((a for a in APPS if a['name'] == app_name), None)
        if app_cfg is None:
            continue
        print(f'\nScraping tambahan {EXTRA_COUNT} review untuk {app_name}...')
        extra = scrape_app_reviews(
            app_name=app_cfg['name'],
            app_id=app_cfg['app_id'],
            count=EXTRA_COUNT,
            max_retry=MAX_RETRY
        )
        new_records.extend(extra)
        time.sleep(DELAY_SECONDS)

    if not new_records:
        print('Tidak ada review tambahan yang berhasil diambil. Menghentikan loop.')
        break

    # Preprocessing review tambahan
    df_extra = pd.DataFrame(new_records)
    df_extra = df_extra[df_extra['text'].notna()]
    df_extra = df_extra[df_extra['text'].str.strip() != '']
    df_extra['_wc'] = df_extra['text'].apply(lambda x: len(str(x).split()))
    df_extra = df_extra[df_extra['_wc'] >= 5].drop(columns=['_wc'])

    tqdm.pandas(desc='Deteksi bahasa (tambahan)')
    df_extra['lang'] = df_extra['text'].progress_apply(detect_language)
    df_extra = df_extra[df_extra['lang'] == 'id']

    df_extra['text_original'] = df_extra['text']
    tqdm.pandas(desc='Normalisasi teks (tambahan)')
    df_extra['text'] = df_extra['text'].progress_apply(normalize_text)

    df_extra['word_count']          = df_extra['text'].apply(lambda x: len(str(x).split()))

    # Gabung dan hapus duplikat
    df = pd.concat([df, df_extra], ignore_index=True)
    df = df.drop_duplicates(subset=['text', 'app'], keep='first').reset_index(drop=True)

    print(f'\nTotal review setelah iterasi {iteration}: {len(df):,}')

print('\n' + '=' * 60)
if len(df) >= MIN_TOTAL:
    print(f'Validasi LULUS: Total review = {len(df):,} (>= {MIN_TOTAL:,})')
else:
    print(f'Validasi GAGAL: Total review = {len(df):,} (< {MIN_TOTAL:,})')
print('=' * 60)


Validasi LULUS: Total review = 12,784 (>= 10,000)


### 2.8 Simpan Hasil Preprocessing

In [35]:
cols_export = [
    'app', 'app_id', 'username', 'rating', 'text', 'text_original',
    'thumbs_up', 'date', 'reply', 'lang', 'word_count',
    'star_sentiment', 'rough_text_sentiment', 'is_mismatch'
]
# Hanya ekspor kolom yang ada
cols_export = [c for c in cols_export if c in df.columns]

df_clean = df[cols_export].copy()
df_clean.to_csv('clean_reviews.csv', index=False, encoding='utf-8-sig')

print(f'Data bersih berhasil disimpan ke clean_reviews.csv')
print(f'Shape final DataFrame : {df_clean.shape}')
print('\nContoh data final (3 baris):')
display(df_clean.head(3))

Data bersih berhasil disimpan ke clean_reviews.csv
Shape final DataFrame : (12784, 11)

Contoh data final (3 baris):


,app,app_id,username,rating,text,text_original,thumbs_up,date,reply,lang,word_count
0,Gojek,com.gojek.app,Pengguna Google,5,saya suka gojek dan semua dr aplikasi ini semo...,saya suka gojek dan semua dr aplikasi ini semo...,0,2026-05-30 16:59:41,None,id,13
1,Gojek,com.gojek.app,Pengguna Google,1,"ga pemilik aplikasi nya ,ga pengiklan nya ,ga ...","ga pemilik aplikasi nya ,ga pengiklan nya ,ga ...",1,2026-05-30 16:41:25,None,id,14
2,Gojek,com.gojek.app,Pengguna Google,5,"katanya potongan cuma 8%,ke driver. gw dapet o...","katanya potongan cuma 8%,ke driver. gw dapet o...",1,2026-05-30 16:39:05,None,id,22


## Bagian 3: Eksplorasi Data

### 3.1 Distribusi Rating per Aplikasi

In [36]:
print('DISTRIBUSI RATING PER APLIKASI')
print('=' * 60)

rating_dist = df_clean.groupby(['app', 'rating']).size().unstack(fill_value=0)
rating_dist.columns = [f'Rating {c}' for c in rating_dist.columns]
rating_dist['Total'] = rating_dist.sum(axis=1)

display(rating_dist)

print('\nRata-rata rating per aplikasi:')
avg_rating = df_clean.groupby('app')['rating'].mean().round(2)
print(avg_rating.to_string())

DISTRIBUSI RATING PER APLIKASI


,Rating 1,Rating 2,Rating 3,Rating 4,Rating 5,Total
app,,,,,,
BCA Mobile,1648,360,309,175,598,3090
DANA,745,158,153,132,837,2025
Gojek,880,162,145,142,799,2128
Shopee,774,141,127,131,1338,2511
Tokopedia,1807,233,226,141,623,3030



Rata-rata rating per aplikasi:
app
BCA Mobile    2.26
DANA          3.08
Gojek         2.91
Shopee        3.45
Tokopedia     2.19


### 3.2 Rata-rata Word Count

In [37]:
def text_segment(wc):
    if wc < 20:
        return 'Pendek (< 20 kata)'
    elif wc <= 50:
        return 'Sedang (20-50 kata)'
    else:
        return 'Panjang (> 50 kata)'

df_clean['text_segment'] = df_clean['word_count'].apply(text_segment)

print('RATA-RATA WORD COUNT PER SEGMEN TEKS')
print('=' * 60)

segment_stats = df_clean.groupby('text_segment').agg(
    Jumlah_Review=('word_count', 'count'),
    Rata_rata_Word_Count=('word_count', 'mean'),
    Min_Word_Count=('word_count', 'min'),
    Max_Word_Count=('word_count', 'max')
).round(2)

# Urutkan segmen
segment_order = ['Pendek (< 20 kata)', 'Sedang (20-50 kata)', 'Panjang (> 50 kata)']
segment_stats = segment_stats.reindex([s for s in segment_order if s in segment_stats.index])

display(segment_stats)

print('\nDistribusi segmen teks:')
seg_dist = df_clean['text_segment'].value_counts()
for seg in segment_order:
    if seg in seg_dist:
        pct = seg_dist[seg] / len(df_clean) * 100
        print(f'  {seg:<25}: {seg_dist[seg]:,} review ({pct:.1f}%)')

RATA-RATA WORD COUNT PER SEGMEN TEKS


,Jumlah_Review,Rata_rata_Word_Count,Min_Word_Count,Max_Word_Count
text_segment,,,,
Pendek (< 20 kata),8481,10.32,4,19
Sedang (20-50 kata),3552,29.85,20,50
Panjang (> 50 kata),751,66.86,51,98



Distribusi segmen teks:
  Pendek (< 20 kata)       : 8,481 review (66.3%)
  Sedang (20-50 kata)      : 3,552 review (27.8%)
  Panjang (> 50 kata)      : 751 review (5.9%)


### 3.3 Ringkasan Akhir Dataset

In [38]:
print('RINGKASAN AKHIR DATASET')
print('=' * 60)
print(f'Total review final           : {len(df_clean):,}')
print(f'Jumlah aplikasi              : {df_clean["app"].nunique()}')
print(f'Rentang rating               : {df_clean["rating"].min()} - {df_clean["rating"].max()}')
print(f'Rata-rata word count         : {df_clean["word_count"].mean():.2f} kata')
print()
print('Distribusi per aplikasi:')
print(df_clean['app'].value_counts().to_string())
print()
print('File yang dihasilkan:')
print('  - raw_reviews.csv   : data mentah dari scraping')
print('  - clean_reviews.csv : data bersih siap untuk pemodelan')

RINGKASAN AKHIR DATASET
Total review final           : 12,784
Jumlah aplikasi              : 5
Rentang rating               : 1 - 5
Rata-rata word count         : 19.06 kata

Distribusi per aplikasi:
app
BCA Mobile    3090
Tokopedia     3030
Shopee        2511
Gojek         2128
DANA          2025

File yang dihasilkan:
  - raw_reviews.csv   : data mentah dari scraping
  - clean_reviews.csv : data bersih siap untuk pemodelan
